# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [8]:
import json
import tiktoken
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

##  LLM

In [9]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ollama import Client


load_dotenv()

True

In [10]:
api_key = os.getenv("OLLAMA_API_KEY")

if not api_key:
    raise ValueError("OLLAMA_API_KEY absente du fichier .env")

llm = ChatOllama(
    model="gemma4:31b",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {"Authorization": f"Bearer {api_key}"}
    },
    temperature=0,
)

print(llm.invoke("Bonjour").content)

Bonjour ! Comment puis-je vous aider aujourd'hui ?


# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [11]:
pdf_file = "./cv.pdf"

In [12]:
pdf_loader = PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [13]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=300,
    chunk_overlap=20
)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [14]:
chunks = pdf_loader.load_and_split(text_splitter)

len(chunks)

4

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

print(len(embedding_model.embed_query("test")))

c:\Users\DELL LATITUDE\OneDrive\Bureau\AIDC\S3\SDI\My_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7120.35it/s]


1024


Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [16]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="cv_v1",
    persist_directory="./store"
)

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [17]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [18]:
#test
retrieved_chunks = retriever.invoke("Présente mon profil professionnel.")
print(retrieved_chunks)
print("nombre de chunks est :", len(retrieved_chunks))


[Document(metadata={'title': '', 'source': './cv.pdf', 'creationdate': '2026-10-15T19:08:34+00:00', 'trapped': '/False', 'keywords': '', 'moddate': '2026-10-15T19:08:34+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'total_pages': 1, 'author': '', 'page_label': '1', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.29', 'page': 0}, page_content='MOUAAD BOUISHAK\nData Engineer | Recherche stage PFE\n♂¶ap-¶arker-altNador, Maroc |♂phone0629821427 |✉bouishak.mouaad@gmail.com |/linkedinLinkedIn |/githubGitHub\nPROFIL\nÉtudiant en dernière année de Master Artificial Intelligence & Digital Computing, spécialisé en Data Engineering. Solides\nbases en ETL et traitement de données batch et streaming, développées à travers des projets end-to-end, notamment sur\nAzure. Compétences en Machine Learning et MLOps. À la recherche d’un stage PFE en Data Engineering.\nEXPÉRIENCE\nData Engineering & Machine Learn

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [19]:
prompt_template = """
Tu es un assistant spécialisé dans la présentation du profil professionnel
du candidat à partir de son CV.

Réponds aux questions concernant ses études, ses compétences techniques,
ses projets, ses expériences et son parcours professionnel.

Règles :
- Utilise uniquement les informations présentes dans le contexte fourni.
- N'invente aucune compétence, expérience, date ou réalisation.
- Si l'information demandée est absente du CV, réponds : "Je ne sais pas !".
- Réponds directement avec une formulation naturelle et précise.
- Réponds dans la langue utilisée par l'utilisateur.

<context>
{context}
</context>

<question>
{question}
</question>
"""

### Note sur le contexte
Vous pouvez ajouter cette précision si vous savez que votre document concerne la Direction des études et des prévisions financières (DEPF). Vous indiquez alors le contexte suivant : *« Le contexte concerne la Note de conjoncture de la Direction des études et des prévisions financières de décembre 2025. »* C'est une phrase utile. Si vous préférez que le prompt reste général, quel que soit le fichier PDF, vous pouvez simplement retirer cette partie.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [20]:
user_input = "Présente mon profil professionnel."

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

'MOUAAD BOUISHAK\nData Engineer | Recherche stage PFE\n♂¶ap-¶arker-altNador, Maroc |♂phone0629821427 |✉bouishak.mouaad@gmail.com |/linkedinLinkedIn |/githubGitHub\nPROFIL\nÉtudiant en dernière année de Master Artificial Intelligence & Digital Computing, spécialisé en Data Engineering. Solides\nbases en ETL et traitement de données batch et streaming, développées à travers des projets end-to-end, notamment sur\nAzure. Compétences en Machine Learning et MLOps. À la recherche d’un stage PFE en Data Engineering.\nEXPÉRIENCE\nData Engineering & Machine Learning Intern\nONEE - Branche Eau\nJuil. 2026 – Août 2026\nNador, Maroc\n• Analyse des données de consommation pour soutenir le suivi des consommations et l’optimisation de la puissance souscrite.\n• Développement et automatisation de pipelines ETL avec Python et SQL Server pour centraliser les données Excel/PDF.\n• Conception d’un dashboard Power BI pour analyser consommation, puissance, dépassements, pénalités et coûts.\n• Application de 

In [21]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [22]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Tu es un assistant spécialisé dans la présentation du profil professionnel
du candidat à partir de son CV.

Réponds aux questions concernant ses études, ses compétences techniques,
ses projets, ses expériences et son parcours professionnel.

Règles :
- Utilise uniquement les informations présentes dans le contexte fourni.
- N'invente aucune compétence, expérience, date ou réalisation.
- Si l'information demandée est absente du CV, réponds : "Je ne sais pas !".
- Réponds directement avec une formulation naturelle et précise.
- Réponds dans la langue utilisée par l'utilisateur.

<context>
MOUAAD BOUISHAK
Data Engineer | Recherche stage PFE
♂¶ap-¶arker-altNador, Maroc |♂phone0629821427 |✉bouishak.mouaad@gmail.com |/linkedinLinkedIn |/githubGitHub
PROFIL
Étudiant en dernière année de Master Artificial Intelligence & Digital Computing, spécialisé en Data Engineering. Solides
bases en ETL et traitement de données batch et streaming, développées à travers des projets end-to-end, notamment su

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [23]:

resp = llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

Mouaad Bouishak est un étudiant en dernière année de Master en Artificial Intelligence & Digital Computing à la Faculté des Sciences et Techniques de Béni Mellal, spécialisé en Data Engineering.

Il possède de solides compétences en ETL et en traitement de données (batch et streaming), avec une expertise particulière sur l'écosystème Azure (ADLS Gen2, Databricks, Synapse). Son profil combine également des compétences en Machine Learning et MLOps.

Son parcours est marqué par :
- **Une expérience professionnelle** à l'ONEE (Branche Eau), où il a automatisé des pipelines ETL avec Python et SQL Server, conçu des dashboards Power BI et développé un modèle de prévision avec XGBoost.
- **Des projets techniques d'envergure**, notamment la création d'une plateforme de traitement de paiements en temps réel via Apache Kafka et Azure, ainsi qu'une plateforme de prévision d'énergie solaire intégrant AWS S3, Snowflake, dbt, MLflow et FastAPI.
- **Un socle technique varié** incluant Python, SQL, Apache Spark, Apache Airflow, Docker et diverses bases de données (PostgreSQL, MongoDB, SQL Server).

Il est actuellement à la recherche d'un stage de fin d'études (PFE) en Data Engineering.

None


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [24]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [25]:
resp = RAG("Présente mon profil professionnel.")
print(display(Markdown(resp)))

Mouaad Bouishak est un étudiant en dernière année de Master en Artificial Intelligence & Digital Computing à la Faculté des Sciences et Techniques de Béni Mellal, spécialisé en Data Engineering.

Il possède de solides bases en ETL et en traitement de données (batch et streaming), ainsi que des compétences en Machine Learning et MLOps. Son parcours est marqué par la réalisation de projets end-to-end, notamment sur Azure.

**Ses compétences techniques incluent :**
*   **Data Engineering & Big Data :** Python, SQL, ETL/ELT, Data Pipelines, Data Lake, Data Warehouse, Apache Kafka, Apache Spark et Apache Airflow.
*   **Cloud & Plateformes :** Azure (ADLS Gen2, Databricks, Synapse), AWS S3, Snowflake, PostgreSQL, SQL Server, MongoDB et Docker.
*   **Machine Learning & Analytics :** scikit-learn, NLP, détection d'anomalies, MLflow, FastAPI, Power BI et Grafana.

**Ses expériences et projets notables :**
*   **Stage chez ONEE (Branche Eau) :** Il a automatisé des pipelines ETL avec Python et SQL Server, conçu un dashboard Power BI et développé un modèle de prévision (XGBoost) pour les besoins en puissance souscrite.
*   **Projet Azure Real-Time Payment Processing :** Mise en place d'une plateforme de traitement de transactions financières en temps réel utilisant Kafka, l'architecture Medallion sur ADLS Gen2, Databricks/PySpark, Azure Synapse et l'orchestration via Airflow.
*   **Projet Solar Energy Forecasting (MLOps) :** Développement d'une plateforme de prévision d'énergie solaire utilisant AWS S3, Snowflake, dbt, PySpark, MLflow et FastAPI.

Il est certifié en Machine Learning (DeepLearning.AI), en Data Analysis (Microsoft) et en Git/GitHub (Udemy). Il maîtrise l'arabe, le français et l'anglais.

None


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [26]:
#question ou query 
user_input = "Présente mon profil professionnel."

Récupérer le context 

In [27]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [28]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [29]:
# Réponse d'un RAG
answer = RAG("Présente mon profil professionnel.")
print(display(Markdown(answer)))

Mouaad Bouishak est un étudiant en dernière année de Master en Artificial Intelligence & Digital Computing à la Faculté des Sciences et Techniques de Béni Mellal, spécialisé en Data Engineering.

Il possède de solides compétences en ETL et en traitement de données (batch et streaming), avec une expertise particulière sur l'écosystème Azure (ADLS Gen2, Databricks, Synapse). Son profil combine également des compétences en Machine Learning et MLOps.

Son parcours est marqué par :
- **Une expérience professionnelle** à l'ONEE (Branche Eau), où il a automatisé des pipelines ETL avec Python et SQL Server, conçu des dashboards Power BI et développé un modèle de prévision basé sur XGBoost.
- **Des projets techniques significatifs**, notamment la création d'une plateforme de traitement de paiements en temps réel sur Azure (utilisant Kafka, PySpark et Airflow) et une plateforme de prévision d'énergie solaire intégrant AWS S3, Snowflake, dbt et MLflow.
- **Un socle technique varié** incluant Python, SQL, Apache Spark, Docker, ainsi que des certifications de DeepLearning.AI, Microsoft et Udemy.

Il est actuellement à la recherche d'un stage de fin d'études (PFE) en Data Engineering.

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [30]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [31]:
# Concernant le juge, il est nécessaire d'employer un modèle plus puissant que celui mis en œuvre par le RAG. 
groundness_checker = ChatOllama(
    model="gemma4:31b",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {"Authorization": f"Bearer {api_key}"}
    },
    temperature=0,
)

In [32]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [33]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

Voici l'évaluation détaillée de la réponse générée par l'IA :

### Étapes d'évaluation

1. **Analyse du contexte :** Identifier toutes les informations disponibles dans le CV fourni (identité, formation, compétences, expériences, projets, certifications et objectif professionnel).
2. **Analyse de la réponse :** Décomposer la réponse de l'IA en affirmations distinctes.
3. **Vérification de la provenance (Cross-referencing) :** Pour chaque affirmation de la réponse, rechercher la preuve correspondante dans le texte du contexte.
4. **Détection d'hallucinations :** Vérifier si l'IA a ajouté des informations extérieures au contexte (connaissances générales ou inventions).
5. **Attribution du score :** Noter la réponse selon la métrique de fidélité stricte au contexte.

---

### Explication étape par étape

*   **Affirmation 1 :** *"Mouaad Bouishak est un étudiant en dernière année de Master en Artificial Intelligence & Digital Computing à la Faculté des Sciences et Techniques de Béni Mellal, spécialisé en Data Engineering."*
    *   **Vérification :** Présent dans les sections "PROFIL" et "FORMATION". **(Conforme)**
*   **Affirmation 2 :** *"Il possède de solides compétences en ETL et en traitement de données (batch et streaming), avec une expertise particulière sur Azure."*
    *   **Vérification :** Présent dans la section "PROFIL". **(Conforme)**
*   **Affirmation 3 :** *"Son profil technique inclut la maîtrise de Python, SQL, Apache Kafka, Spark et Airflow, ainsi que des compétences en Machine Learning et MLOps."*
    *   **Vérification :** Présent dans les sections "PROFIL" et "COMPÉTENCES". **(Conforme)**
*   **Affirmation 4 :** *"Une expérience professionnelle à l'ONEE (Branche Eau), où il a automatisé des pipelines ETL, conçu des dashboards Power BI et développé un modèle de prévision avec XGBoost."*
    *   **Vérification :** Présent dans la section "EXPÉRIENCE". **(Conforme)**
*   **Affirmation 5 :** *"Des projets personnels d'envergure, notamment la création d'une plateforme de traitement de paiements en temps réel sur Azure (architecture Medallion, Databricks, Synapse) et une plateforme de prévision d'énergie solaire intégrant AWS S3, Snowflake, dbt et MLflow."*
    *   **Vérification :** Présent dans la section "PROJETS PERSONNELS". **(Conforme)**
*   **Affirmation 6 :** *"Il est certifié en Machine Learning (DeepLearning.AI), en analyse de données (Microsoft) et sur Git/GitHub (Udemy)."*
    *   **Vérification :** Présent dans la section "CERTIFICATIONS". **(Conforme)**
*   **Affirmation 7 :** *"Il est actuellement à la recherche d'un stage PFE en Data Engineering."*
    *   **Vérification :** Présent dans la section "PROFIL". **(Conforme)**

L'IA a synthétisé les informations sans ajouter aucun élément extérieur. Elle a respecté scrupuleusement les données fournies dans le contexte.

---

### Évaluation et Score

La réponse est entièrement dérivée du contexte. Aucune information externe n'a été ajoutée, et aucune information du contexte n'a été déformée.

**Score : 5 - La métrique est entièrement respectée**

None


### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
